# 01 — Schema Design


> **Note.** This notebook is about schema design — the concepts apply equally to the cassette-native `InfonStore` and to the legacy `InfonEngine` class. The code below uses the legacy API for continuity with pre-cassette tests. For the cassette-native version of the same ideas (plus the `extraction_report` diagnostic that catches silent failures), see **[00 — Quick Start](00_quick_start.ipynb)**.

The schema is the lens through which infon reads documents. Every anchor you define
is a concept the system can detect — and every concept you omit is invisible.

This notebook covers:
- How anchor types map to triple roles
- Token selection strategy
- Hierarchy metadata
- Multi-domain schemas
- How to iterate on your schema

In [ ]:
import json
from pathlib import Path
from infon import InfonEngine, InfonConfig
from infon.schema import AnchorSchema
from infon.encoder import Encoder

## Type → Role mapping

Every infon is a triple `<<predicate, subject, object>>`. Anchor types determine which role an anchor fills:

```
actor    → subject     (who does the action)
relation → predicate   (what action is taken)
*        → object      (everything else: features, markets, locations, etc.)
```

This means the schema design directly controls what triples you get. A schema with only actors
and features but no relations will produce nothing — there's no predicate to form a triple.

## Token selection

Each anchor's `tokens` list tells the AnchorProjector which BERT vocabulary entries to watch.
The projector maps each token to BERT vocab IDs and takes the **max** across them.

Good tokens are:
- **Lexical variants**: `["invest", "investment", "investing"]`
- **Synonyms**: `["launch", "unveil", "introduce", "release"]`
- **Domain-specific forms**: `["solid-state", "solid state"]`

Bad tokens:
- **Too generic**: `["thing", "do", "make"]` — activates on everything
- **Too rare**: misspellings or jargon not in BERT's vocabulary
- **Overlapping**: if two anchors share tokens, they'll always co-activate

In [ ]:
# Let's see how the projector resolves tokens to BERT vocab IDs
schema = AnchorSchema({
    "toyota":  {"type": "actor", "tokens": ["toyota"]},
    "invest":  {"type": "relation", "tokens": ["invest", "investment", "investing"]},
    "battery": {"type": "feature", "tokens": ["battery", "batteries"]},
})

encoder = Encoder(schema=schema)
projector = encoder._projector

for name, ids in projector.anchor_token_ids.items():
    tokens = [projector.tokenizer.convert_ids_to_tokens(i) for i in ids]
    print(f"  {name:12s} → vocab IDs {ids} → tokens {tokens}")

## Raw SPLADE activations

Before projection, SPLADE produces a 30,522-dim sparse vector over the full BERT vocabulary.
Let's see what it does with a test sentence.

In [ ]:
import numpy as np

text = "Toyota announced a $13.5 billion investment in solid-state battery technology."
sparse = encoder.encode_sparse([text])[0]

# Top 20 BERT vocab tokens by activation
top_ids = np.argsort(sparse)[-20:][::-1]
print("Top 20 SPLADE activations:")
for tid in top_ids:
    token = encoder.tokenizer.convert_ids_to_tokens(int(tid))
    print(f"  {token:20s}  {sparse[tid]:.3f}")

In [ ]:
# Now see how projection maps these to our anchors
activations = encoder.encode_single(text)
print("\nAnchor activations after projection:")
for name, score in sorted(activations.items(), key=lambda x: -x[1]):
    print(f"  {name:12s}  {score:.3f} {'█' * int(score * 15)}")

## Hierarchy metadata

Anchors can carry metadata that enriches the extracted infons. This metadata flows through
to the `subject_meta`, `predicate_meta`, and `object_meta` fields on each infon.

In [ ]:
hierarchical_schema = {
    # Actors with organisation metadata
    "toyota": {
        "type": "actor",
        "tokens": ["toyota"],
        "organisation_type": "private-sector",
        "country_code": "JP",
        "canonical_name": "Toyota Motor Corporation",
    },
    "tesla": {
        "type": "actor",
        "tokens": ["tesla"],
        "organisation_type": "private-sector",
        "country_code": "US",
        "canonical_name": "Tesla Inc.",
    },

    # Relations with domain labels
    "invest": {
        "type": "relation",
        "tokens": ["invest", "investment"],
        "domain": "financial",
        "level": "strategic",
    },
    "partner": {
        "type": "relation",
        "tokens": ["partner", "partnership", "collaborate"],
        "domain": "business",
        "level": "strategic",
    },

    # Markets with geographic hierarchy
    "us": {
        "type": "market",
        "tokens": ["us", "united states"],
        "level": "country",
        "macro_region": "North America",
        "country_code": "US",
    },
    "north_america": {
        "type": "market",
        "tokens": ["north america"],
        "level": "region",
        "macro_region": "North America",
    },

    # Features with parent-child hierarchy
    "ev": {
        "type": "feature",
        "tokens": ["ev", "electric vehicle"],
        "level": "category",
    },
    "battery": {
        "type": "feature",
        "tokens": ["battery", "batteries"],
        "level": "component",
        "parent": "ev",
    },
    "solid_state": {
        "type": "feature",
        "tokens": ["solid-state", "solid state"],
        "level": "technology",
        "parent": "battery",
    },
}

schema_h = AnchorSchema(hierarchical_schema)

# Traverse the hierarchy
print("Feature hierarchy:")
print(f"  solid_state ancestors: {schema_h.get_ancestors('solid_state')}")
print(f"  ev descendants:        {schema_h.get_descendants('ev')}")
print()
print(f"  us hierarchy:    {schema_h.get_hierarchy('us')}")
print(f"  toyota hierarchy: {schema_h.get_hierarchy('toyota')}")

## Domain examples

The same four-type pattern works across domains. Here are starter schemas for different fields:

In [ ]:
# Geopolitical domain
geopolitical_schema = {
    "us_gov":     {"type": "actor", "tokens": ["united states", "washington", "us government"]},
    "china_gov":  {"type": "actor", "tokens": ["china", "beijing", "chinese government"]},
    "russia":     {"type": "actor", "tokens": ["russia", "moscow", "kremlin"]},
    "nato":       {"type": "actor", "tokens": ["nato", "alliance"]},
    
    "sanction":   {"type": "relation", "tokens": ["sanction", "sanctions", "embargo"]},
    "negotiate":  {"type": "relation", "tokens": ["negotiate", "negotiation", "talks", "diplomacy"]},
    "deploy":     {"type": "relation", "tokens": ["deploy", "deployment", "station"]},
    "trade":      {"type": "relation", "tokens": ["trade", "tariff", "import", "export"]},
    
    "military":   {"type": "feature", "tokens": ["military", "defense", "armed forces"]},
    "nuclear":    {"type": "feature", "tokens": ["nuclear", "atomic"]},
    "energy":     {"type": "feature", "tokens": ["energy", "oil", "gas", "pipeline"]},
    
    "europe":     {"type": "market", "tokens": ["europe", "european"]},
    "pacific":    {"type": "market", "tokens": ["pacific", "indo-pacific", "asia"]},
}

print(f"Geopolitical schema: {len(geopolitical_schema)} anchors")
for atype in ["actor", "relation", "feature", "market"]:
    names = [n for n, v in geopolitical_schema.items() if v["type"] == atype]
    print(f"  {atype:10s}: {', '.join(names)}")

In [ ]:
# Clinical domain
clinical_schema = {
    "patient":    {"type": "actor", "tokens": ["patient", "subject"]},
    "physician":  {"type": "actor", "tokens": ["physician", "doctor", "clinician"]},
    "fda":        {"type": "actor", "tokens": ["fda", "food and drug administration"]},
    
    "prescribe":  {"type": "relation", "tokens": ["prescribe", "administer", "dose"]},
    "diagnose":   {"type": "relation", "tokens": ["diagnose", "diagnosis", "detect"]},
    "approve":    {"type": "relation", "tokens": ["approve", "approval", "authorize"]},
    "trial":      {"type": "relation", "tokens": ["trial", "study", "clinical trial"]},
    
    "diabetes":   {"type": "feature", "tokens": ["diabetes", "diabetic", "glucose"]},
    "oncology":   {"type": "feature", "tokens": ["cancer", "oncology", "tumor"]},
    "cardio":     {"type": "feature", "tokens": ["cardiac", "heart", "cardiovascular"]},
    "immunotherapy": {"type": "feature", "tokens": ["immunotherapy", "immune"]},
}

print(f"Clinical schema: {len(clinical_schema)} anchors")

In [ ]:
# Supply chain domain
supply_chain_schema = {
    "supplier":   {"type": "actor", "tokens": ["supplier", "vendor", "manufacturer"]},
    "carrier":    {"type": "actor", "tokens": ["carrier", "shipper", "logistics"]},
    "warehouse":  {"type": "actor", "tokens": ["warehouse", "distribution center", "fulfillment"]},
    
    "ship":       {"type": "relation", "tokens": ["ship", "deliver", "transport"]},
    "delay":      {"type": "relation", "tokens": ["delay", "disruption", "shortage"]},
    "order":      {"type": "relation", "tokens": ["order", "procure", "purchase"]},
    
    "semiconductor": {"type": "feature", "tokens": ["semiconductor", "chip", "microchip"]},
    "raw_material":  {"type": "feature", "tokens": ["raw material", "commodity", "lithium", "cobalt"]},
    
    "asia":       {"type": "market", "tokens": ["asia", "asian"]},
    "americas":   {"type": "market", "tokens": ["americas", "north america", "south america"]},
}

print(f"Supply chain schema: {len(supply_chain_schema)} anchors")

## Schema iteration: activation debugging

The most important thing you can do is test your schema against real text.
Encode sample sentences, check what activates, and adjust tokens.

In [ ]:
# Test the geopolitical schema against sample text
geo_schema = AnchorSchema(geopolitical_schema)
geo_encoder = Encoder(schema=geo_schema)

test_sentences = [
    "The United States imposed sanctions on Russian energy exports.",
    "NATO deployed additional forces to the Baltic states.",
    "China and the US resumed trade negotiations in Geneva.",
]

for sent in test_sentences:
    acts = geo_encoder.encode_single(sent)
    top = sorted(acts.items(), key=lambda x: -x[1])[:5]
    print(f"\n\"{sent}\"")
    for name, score in top:
        atype = geopolitical_schema[name]["type"]
        print(f"  {atype:10s} {name:15s} {score:.3f}")

## Guidelines

1. **Start small** — 15-25 anchors. Add more once you see what's missing.
2. **Balance types** — you need actors, relations, AND objects. No predicates = no triples.
3. **Be specific** — `"solid_state"` is better than `"technology"` for domain precision.
4. **Include variants** — `["invest", "investment", "investing"]` catches more.
5. **Test on real text** — encode sample sentences and check activations.
6. **Use hierarchy** — `parent` links enable hierarchical grounding support.

---

**Next:** [02 Ingestion & Extraction](02_ingestion.ipynb) — what happens inside `ingest()`.